In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.metrics import f1_score

In [10]:
def kl_divergence(p, q):
    eps = 1e-12
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    return np.sum(p * np.log(p / q))

In [11]:
def sqrt_js_divergence(p, q):
    p = p.astype(float)
    q = q.astype(float)
    
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    
    js = 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)
    js = max(js, 0.0)
    return np.sqrt(js)

In [12]:
ms = pd.read_excel(r"C:\Users\user\Desktop\디어젠전달용\SMILES_pred\smiles_122.xlsx")
ms["m/z"] = ms["m/z"].astype(int)
ms["intensity"] = ms["intensity"].astype(float)

names = ms["Name"].drop_duplicates().reset_index(drop=True)
namesidx = {n: idx for idx, n in enumerate(names)}
ms["m/z_idx"] = (np.round(ms["m/z"]) - 1).astype(int)
ms["row_idx"] = ms["Name"].map(namesidx)

In [13]:
ms

,Name,Form,m/z,intensity,SMILES,테러 물질 여부,m/z_idx,row_idx
0,Formaldehyde,CH2O,12,10.00,C=O,X,11,0
1,Formaldehyde,CH2O,13,10.00,C=O,X,12,0
2,Formaldehyde,CH2O,14,10.00,C=O,X,13,0
3,Formaldehyde,CH2O,15,20.00,C=O,X,14,0
4,Formaldehyde,CH2O,28,240.00,C=O,X,27,0
...,...,...,...,...,...,...,...,...
3962,Glufosinate,C5H12NO4P,135,85.61,CP(=O)(CCC(C(=O)O)N)O,X,134,121
3963,Glufosinate,C5H12NO4P,136,709.59,CP(=O)(CCC(C(=O)O)N)O,X,135,121
3964,Glufosinate,C5H12NO4P,162,17.08,CP(=O)(CCC(C(=O)O)N)O,X,161,121
3965,Glufosinate,C5H12NO4P,163,16.28,CP(=O)(CCC(C(=O)O)N)O,X,162,121


In [14]:
N = ms["row_idx"].max() + 1
D = ms["m/z_idx"].max() + 1

data = np.zeros((N, D))
grouped = ms.groupby(["row_idx", "m/z_idx"])["intensity"].max()

row_idx = grouped.index.get_level_values(0).to_numpy()
col_idx = grouped.index.get_level_values(1).to_numpy()
values = grouped.to_numpy()

data[row_idx, col_idx] = values

In [15]:
ter_label = ms[["Name", "테러 물질 여부"]].drop_duplicates().reset_index(drop=True)

In [20]:
# row_idx별 대표 Name/SMILES 만들기
meta = (ms[["row_idx", "Name"] + (["SMILES"] if "SMILES" in ms.columns else [])]
        .drop_duplicates(subset=["row_idx"])
        .sort_values("row_idx")
        .reset_index(drop=True))
meta

,row_idx,Name,SMILES
0,0,Formaldehyde,C=O
1,1,Methyl hydrazine,CNN
2,2,Formic acid,C(=O)O
3,3,Methyl chloride,CCl
4,4,Methylamine,CN
...,...,...,...
117,117,Acetaminophen,CC(=O)NC1=CC=C(C=C1)O
118,118,Aspirin,CC(=O)OC1=CC=CC=C1C(=O)O
119,119,Paraquat dichloride,C[N+]1=CC=C(C=C1)C2=CC=[N+](C=C2)C.[Cl-].[Cl-]
120,120,N-(Phosphonomethyl)glycine,C(C(=O)O)NCP(=O)(O)O


In [24]:
noise_set = [0.03, 0.05, 0.10]
case_set = [[1, 1], [1, -1], [-1, 1], [-1, -1]]

noise = noise_set[0]
case = case_set[0]

avg_acc = np.zeros(len(noise_set))


for k in range(N):
    noise_data = data[k].copy()
    
    idx = data[k].argsort()
    peak2 = idx[-2]
    peak3 = idx[-3]
    noise_data[peak2] = data[k][peak2] * (1 + (noise * case[0]))
    noise_data[peak3] = data[k][peak3] * (1 + (noise * case[1]))
    
    noise_data /= noise_data.sum()
    
    jsd = np.zeros(N)
    
    for l in range(N):
        jsd[l] = sqrt_js_divergence(noise_data, data[l])
        
    pred = jsd.argmin()

    # 출력
    name_k = meta.loc[k, "Name"]
    name_p = meta.loc[pred, "Name"]

    if "SMILES" in meta.columns:
        smi_k = meta.loc[k, "SMILES"]
        smi_p = meta.loc[pred, "SMILES"]
        print(f"{name_k} ({smi_k})  -->  pred={pred}: {name_p} ({smi_p})")
    else:
        print(f"[k={k}] {name_k}  -->  pred={pred}: {name_p}")

Formaldehyde (C=O)  -->  pred=0: Formaldehyde (C=O)
Methyl hydrazine ( CNN )  -->  pred=1: Methyl hydrazine ( CNN )
Formic acid (C(=O)O  )  -->  pred=2: Formic acid (C(=O)O  )
Methyl chloride ( CCl  )  -->  pred=3: Methyl chloride ( CCl  )
Methylamine (CN )  -->  pred=4: Methylamine (CN )
Hydrogen cyanide (C#N )  -->  pred=5: Hydrogen cyanide (C#N )
Vinyl chloride (C=CCl )  -->  pred=6: Vinyl chloride (C=CCl )
Ethylene oxide (C1CO1 )  -->  pred=7: Ethylene oxide (C1CO1 )
Trimethylamine (CN(C)C  )  -->  pred=8: Trimethylamine (CN(C)C  )
Propylene oxide (CC1CO1)  -->  pred=9: Propylene oxide (CC1CO1)
Methyl ethyl ketone (CCC(=O)C)  -->  pred=10: Methyl ethyl ketone (CCC(=O)C)
Methyl vinyl ketone (CC(=O)C=C )  -->  pred=11: Methyl vinyl ketone (CC(=O)C=C )
Acrylic acid (C=CC(=O)O )  -->  pred=12: Acrylic acid (C=CC(=O)O )
Methyl acrylate (COC(=O)C=C )  -->  pred=13: Methyl acrylate (COC(=O)C=C )
Nitrobenzene (C1=CC=C(C=C1)[N+](=O)[O-])  -->  pred=14: Nitrobenzene (C1=CC=C(C=C1)[N+](=O)[O-